# 🏠 Feng-Shui-Zimmerplaner in Google Colab

Dieses Notebook führt das Projekt `fengshui_zimmer` komplett im Browser aus – ohne Installation auf deinem PC.

**Ablauf:** Zellen von oben nach unten mit ▶ ausführen (oder `Strg+Enter`).

| Schritt | Was passiert |
|---|---|
| 1 | Projekt laden (GitHub, ZIP-Upload oder Google Drive) |
| 2 | Pakete installieren |
| 3 | Dein Zimmer eintragen |
| 4 | Optimieren → 3 Grundriss-Varianten |
| 5 | Ausstattungs-Check |
| 6 | Foto hochladen → YOLO erkennt Möbel → Empfehlungen |
| 7 | Machine Learning: Vorschläge bewerten, persönliche Gewichte lernen |
| 8 | Ergebnisse sichern |

> ⚠️ Colab löscht alle Dateien, wenn die Sitzung endet. Nutze **Option C (Google Drive)**, wenn deine Bewertungen und Änderungen erhalten bleiben sollen – oder lade sie in Schritt 8 herunter.

## 1 · Projekt laden
Wähle **eine** Option: `QUELLE = "github"`, `"zip"` oder `"drive"`.

- **github**: `GITHUB_URL` auf dein Repo setzen. Bei einem *privaten* Repo brauchst du einen Token (siehe Hinweis in der Zelle).
- **zip**: Es erscheint ein Upload-Knopf → `fengshui_zimmer.zip` auswählen.
- **drive**: Den Ordner `fengshui_zimmer` vorher in dein Google Drive hochladen (z. B. nach *Meine Ablage/fengshui_zimmer*). Alles bleibt dauerhaft gespeichert. ✅ empfohlen

In [ ]:
QUELLE = "zip"   # "github" | "zip" | "drive"
GITHUB_URL = "https://github.com/DEIN-NAME/fengshui-zimmer.git"
DRIVE_ORDNER = "/content/drive/MyDrive/fengshui_zimmer"

# Privates GitHub-Repo: Token auf github.com → Settings → Developer settings → Personal access tokens erstellen
# und links im Colab-Menü 🔑 (Secrets) als GITHUB_TOKEN speichern. Dann wird er hier automatisch benutzt.

import os, glob, subprocess

def projekt_finden(basis="/content"):
    treffer = sorted(glob.glob(f"{basis}/**/fengshui/__init__.py", recursive=True), key=len)
    treffer = [p for p in treffer if ".ipynb_checkpoints" not in p]
    return os.path.dirname(os.path.dirname(treffer[0])) if treffer else None

if QUELLE == "github":
    url = GITHUB_URL
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        url = url.replace("https://", f"https://{token}@")
    except Exception:
        pass
    ziel = "/content/" + GITHUB_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.exists(ziel):
        subprocess.run(["git", "clone", url, ziel], check=True)
    else:
        subprocess.run(["git", "-C", ziel, "pull"], check=False)
    PROJEKT = projekt_finden(ziel)
elif QUELLE == "zip":
    from google.colab import files
    hochgeladen = files.upload()
    for name in hochgeladen:
        if name.endswith(".zip"):
            subprocess.run(["unzip", "-o", "-q", name, "-d", "/content/projekt"], check=True)
    PROJEKT = projekt_finden("/content/projekt")
elif QUELLE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJEKT = projekt_finden(DRIVE_ORDNER) or DRIVE_ORDNER

assert PROJEKT and os.path.exists(os.path.join(PROJEKT, "fengshui")), "Projektordner mit 'fengshui/' nicht gefunden"
os.chdir(PROJEKT)
import sys
if PROJEKT not in sys.path:
    sys.path.insert(0, PROJEKT)
print("✓ Projekt geladen:", PROJEKT)
print(sorted(os.listdir(PROJEKT)))

## 2 · Pakete installieren
`matplotlib` und `numpy` sind in Colab schon da. `ultralytics` (YOLO) wird nur für die Foto-Analyse gebraucht (~1 Minute).
Tipp: Für schnellere Foto-Erkennung **Laufzeit → Laufzeittyp ändern → T4-GPU** (optional).

In [ ]:
!pip install -q ultralytics
%matplotlib inline
import matplotlib
matplotlib.rcParams["figure.dpi"] = 110
print("✓ bereit")

## 3 · Dein Zimmer eintragen
Diese Zelle **überschreibt** `mein_zimmer.py` mit dem Inhalt unten. Ändere hier Maße, Türen, Fenster und Möbel und führe die Zelle aus.

Koordinaten: (0,0) = linke untere Ecke in der Draufsicht · Wände `unten / oben / links / rechts` · `rot` = Blickrichtung (0 = oben, 90 = rechts, 180 = unten, 270 = links).

In [ ]:
%%writefile mein_zimmer.py
"""
>>> HIER DEIN ZIMMER EINTRAGEN <<<

Miss dein Zimmer aus (in Metern) und trage Türen, Fenster und Möbel ein.
Koordinaten: Ursprung (0,0) = linke untere Ecke in der Draufsicht.
    Wände: "unten" (y=0), "oben" (y=tiefe), "links" (x=0), "rechts" (x=breite)
    Öffnung `von` = Abstand vom Wandanfang (unten/oben: von links gemessen, links/rechts: von unten gemessen)

Möbelmaße: `breite` = quer zur Blickrichtung, `tiefe` = in Blickrichtung
           (beim Bett ist `tiefe` die Bettlänge, z.B. 2.0 m).
"""
from fengshui.modell import Balken, Bewohner, Layout, Moebel, Oeffnung, Raum, Schraege


def erstelle_layout() -> Layout:
    raum = Raum(
        name="Mein Zimmer",
        breite=3.6,
        tiefe=4.2,
        hoehe=2.5,
        raumtyp="schlafzimmer",      # schlafzimmer | buero | wohnzimmer | esszimmer | kinderzimmer | studio
        nord_wand="oben",            # welche Wand zeigt nach Norden? (None, wenn unbekannt)
        oeffnungen=[
            Oeffnung("tuer", wand="unten", von=0.2, breite=0.85),
            Oeffnung("fenster", wand="oben", von=1.0, breite=1.4, bruestung=0.9),
            Oeffnung("fenster", wand="rechts", von=2.2, breite=1.0, bruestung=0.9),
        ],
        balken=[
            # Balken(x1, y1, x2, y2)  – z.B. Balken(0, 2.0, 3.6, 2.2)
        ],
        schraege=None,               # z.B. Schraege(wand="links", hoehe_niedrig=1.2, laenge=1.5)
        altbau=False,
        aussenwaende=["oben", "rechts"],
    )

    moebel = [
        Moebel("bett", breite=1.4, tiefe=2.0, hoehe=0.5, name="Bett"),
        Moebel("nachttisch", breite=0.45, tiefe=0.4, hoehe=0.5, name="Nachttisch L",
               gehoert_zu="Bett", seite="links"),
        Moebel("nachttisch", breite=0.45, tiefe=0.4, hoehe=0.5, name="Nachttisch R",
               gehoert_zu="Bett", seite="rechts"),
        Moebel("schreibtisch", breite=1.2, tiefe=0.6, hoehe=0.75, name="Schreibtisch"),
        Moebel("schrank", breite=1.5, tiefe=0.6, hoehe=2.2, name="Kleiderschrank"),
        Moebel("pflanze", breite=0.4, tiefe=0.4, hoehe=1.0, name="Monstera", rund=True),
        Moebel("spiegel", breite=0.6, tiefe=0.04, hoehe=1.6, name="Spiegel"),
        # Beispiele für weitere Möbel:
        # Moebel("kommode", breite=1.0, tiefe=0.45, hoehe=0.8, name="Kommode"),
        # Moebel("regal", breite=0.8, tiefe=0.35, hoehe=1.0, name="Regal"),
        # Moebel("bild", breite=0.8, tiefe=0.03, name="Bild"),
        # Moebel("bank", breite=1.2, tiefe=0.4, hoehe=0.45, name="Bank", gehoert_zu="Bett", seite="vorne"),
        # Moebel("sofa", breite=2.0, tiefe=0.9, name="Sofa"),
        # Moebel("couchtisch", breite=1.0, tiefe=0.6, hoehe=0.4, name="Couchtisch",
        #        gehoert_zu="Sofa", seite="vorne", abstand=0.45),
        # Fest stehende Möbel (Einbauschrank) mit fest=True und x/y/rot angeben:
        # Moebel("schrank", breite=2.0, tiefe=0.6, hoehe=2.4, name="Einbau", x=1.0, y=3.9, rot=180, fest=True),
    ]

    bewohner = Bewohner(
        paar=False,
        sucht_partner=True,
        viele_bildschirme=False,
        arbeit_wichtiger_als_schlaf=False,
        pflanzen_sind_freunde=True,
    )
    return Layout(raum, moebel, bewohner)


# Für den Ausstattungs-Check (Farben, Licht, Bett, Ordnung …)
PROFIL = {
    "wandfarbe": "weiss",
    "akzentfarben": ["grün", "sand"],
    "lichtfarbe_kelvin": [2700, 4000],
    "leuchtmittel": "opal",
    "anzahl_lichtquellen": 2,
    "pendelleuchte_ueber_bett": False,
    "bettkopfteil": "gepolstert",
    "wasserbett": False,
    "bettwaesche_farbe": "hellgrau",
    "bett_unterbau_vollgestopft": False,
    "kleiderschrank": "freistehend",
    "kleiderschrank_vollgestopft": False,
    "uhr_im_raum": False,
    "smartphone_am_bett": True,
    "computer_im_raum": True,
    "pflanzenarten": ["monstera"],
    "kunstpflanzen": False,
    "notizzettel_an_wand": False,
    "unordnung": False,
    "persoenlichkeit": "ruhig",
}

## 4 · Optimieren
Sucht automatisch gute Anordnungen und zeigt die 3 besten Varianten. Dauert ca. 30–60 Sekunden.
Steht in `mein_zimmer.py` schon eine Position (x/y) bei deinen Möbeln, wird zuerst deine **jetzige** Einrichtung bewertet.

In [ ]:
ITERATIONEN = 3000
NEUSTARTS = 6

import importlib, mein_zimmer
importlib.reload(mein_zimmer)
from fengshui.bewertung import bericht
from fengshui.ml import Personalisierung
from fengshui.optimierer import optimiere
from fengshui.visualisierung import zeichne, zeichne_mehrere

layout = mein_zimmer.erstelle_layout()
gewichte = Personalisierung().gewichte
if gewichte:
    print("→ verwende deine persönlich gelernten Gewichte\n")

if any(m.x or m.y for m in layout.moebel):
    print(bericht(layout, gewichte=gewichte))
    zeichne(layout, "Jetzige Einrichtung")

ergebnisse = optimiere(layout, iterationen=ITERATIONEN, neustarts=NEUSTARTS, gewichte=gewichte)
os.makedirs("ausgabe", exist_ok=True)
zeichne_mehrere(ergebnisse, datei="ausgabe/alle_varianten.png", zeigen=True)
for i, (lay, erg) in enumerate(ergebnisse, 1):
    print(f"\n################ VARIANTE {i} ################")
    print(bericht(lay, erg))

## 5 · Ausstattungs-Check (Farben, Licht, Bett, Pflanzen, Ordnung)
Nutzt das `PROFIL` aus Schritt 3.

In [ ]:
from fengshui.ausstattung import drucke_ausstattung
print(drucke_ausstattung(layout.raum, mein_zimmer.PROFIL))

## 6 · Foto-Analyse mit YOLO
Ausführen → Upload-Knopf → Foto deines Zimmers wählen (am besten aus der Tür heraus, Weitwinkel, hell).
Beim ersten Mal wird das YOLOE-Modell heruntergeladen.

In [ ]:
RAUMTYP = "schlafzimmer"      # schlafzimmer | buero | wohnzimmer | esszimmer | kinderzimmer | studio
MODELL = "yoloe-11s-seg.pt"   # genauer: "yoloe-11m-seg.pt"
CONF = 0.2

from google.colab import files
from IPython.display import Image, display
from fengshui.foto import (erkenne, foto_empfehlungen, speichere_erkennungen, text_bericht,
                           zeichne_erkennungen, zu_moebelliste, zu_profil)

fotos = list(files.upload())
os.makedirs("ausgabe", exist_ok=True)
for foto in fotos:
    erk, groesse = erkenne(foto, MODELL, CONF)
    ziel = f"ausgabe/erkennung_{os.path.splitext(foto)[0]}.jpg"
    zeichne_erkennungen(foto, erk, ziel)
    speichere_erkennungen(erk, groesse, f"ausgabe/erkennungen_{os.path.splitext(foto)[0]}.json")
    display(Image(ziel, width=700))
    print(text_bericht(erk, foto_empfehlungen(erk, groesse, RAUMTYP)))
    print("\n=== Möbelliste für Schritt 3 (Standardmaße – bitte nachmessen!) ===")
    for m in zu_moebelliste(erk):
        extra = f', gehoert_zu="{m.gehoert_zu}", seite="{m.seite}"' if m.gehoert_zu else ""
        print(f'    Moebel("{m.typ}", breite={m.breite}, tiefe={m.tiefe}, hoehe={m.hoehe}, name="{m.name}"{extra}),')

## 7 · Machine Learning: Vorschläge bewerten
Es erscheinen nacheinander 8 Grundrisse. Gib unter jedem eine Note **0–10** ins Eingabefeld ein (Enter).
Leer = überspringen, `q` = fertig. Ab 3 Bewertungen wird trainiert; 8–15 sind ideal.
Danach Schritt 4 erneut ausführen – der Optimierer nutzt dann deine Gewichte.

In [ ]:
import importlib, lernen
importlib.reload(mein_zimmer); importlib.reload(lernen)
lernen.ANZAHL_VORSCHLAEGE = 8
lernen.main()

## 8 · Ergebnisse sichern
Lädt `ausgabe/`, `mein_zimmer.py` und `meine_bewertungen.json` als ZIP herunter.
(Bei **Option C / Drive** ist alles schon dauerhaft in deinem Drive gespeichert.)

In [ ]:
import shutil
from google.colab import files
paket = "/content/fengshui_ergebnisse"
os.makedirs(paket, exist_ok=True)
for f in ["mein_zimmer.py", "meine_bewertungen.json"]:
    if os.path.exists(f):
        shutil.copy(f, paket)
if os.path.exists("ausgabe"):
    shutil.copytree("ausgabe", os.path.join(paket, "ausgabe"), dirs_exist_ok=True)
shutil.make_archive("/content/fengshui_ergebnisse", "zip", paket)
files.download("/content/fengshui_ergebnisse.zip")

### Optional: Änderungen zurück zu GitHub pushen
Nur bei `QUELLE = "github"` und gespeichertem `GITHUB_TOKEN` (Schritt 1). Fotos und Bewertungen bleiben wegen `.gitignore` draußen.

In [ ]:
NACHRICHT = "Zimmer-Maße aktualisiert"
!git config user.email "du@example.com" && git config user.name "Colab"
!git add -A && git commit -m "{NACHRICHT}" && git push